# D3.2 · Admission rules — what the investigating agent may touch

**Function D — The Agentic SOC → Understand — Correlation, Intel and the Hunt**

Builds on **[D3.1 · From alert queue to loop operator](https://spbreed.github.io/cyber-commons/lessons/D3.1.html)**.

| | |
|---|---|
| Tools used | OPA |

## What this lesson is

**What it covers.** Admission rules for an investigating agent: which sources it may reach, which fields it may never see, how much it may pull, and why every refusal is recorded with its query.

**Why a security engineer needs it.** Incident response grants the broadest read access in the organisation, at the moment of least supervision, often to an agent. That grant is frequently larger than the incident being investigated. Bounding it per investigation class, in advance, is the difference between a response and a second breach — and the refusal log is the evidence you were on the right side of that.

| | |
|---|---|
| **Day 0 — why** | Incident response grants the broadest read in the organisation at the moment of least supervision — and that grant is often larger than the incident. |
| **Day 1 — how** | Bound the investigating agent per investigation class before it runs: sources, forbidden fields, volume cap. |
| **Day 2 — measure** | This one spends time on purpose. Measure refusals logged with their query — the evidence you stayed on the right side of the line. |

## 1 · The hook

The investigating agent is granted broad read across production so it can find the problem. Broad read across production is frequently what the problem was. Admission rules are how the response avoids becoming the second incident.

> **At CyberTravels.** The admission set is written for CyberTravels' agent-misuse class: agent traces, gateway logs and the tool audit are in; the bookings database, with its payment cards, is not. The refused query that matters is the one asking for ninety thousand rows of a source that IS admitted — CyberTravels' whole gateway log, which is a copy.

## 2 · The framework

```
   investigation class: agent-misuse
   +--------------------------------------------------+
   |  sources   agent.traces  gateway.logs  tool.audit |   allowlist
   |  fields    NOT payment_card, passport_no,         |   denied
   |            message_body                           |
   |  volume    <= 5000 rows                           |   cap
   +--------------------------------------------------+
              |
      query --+--> admitted   -> runs, logged
              |
              +--> refused    -> logged WITH the query text
                               -> a human can grant it deliberately

   the volume cap is the one people leave out: same source, same fields,
   400 rows is an investigation and 90,000 is a copy
```

An investigating agent is handed broad read across the estate so it can find the
problem. Broad read across the estate is frequently what the problem *was*.

Admission rules decide, per investigation class and **before anything runs**,
which sources the agent may reach, which fields it may never see, and how much
it may pull. The volume cap is the one people leave out: the same query over the
same fields is an investigation at four hundred rows and a copy at ninety
thousand.

> **Anchor → D1.0.** This one **spends** the interval on purpose. Bounding the investigator before it runs is slower than handing it broad read, and broad read across the estate is frequently what the incident was — so the time buys not having a second one.

## 3 · Refusals are evidence, not errors

A refused query carries the exact query text, so a human can grant it
deliberately and the grant is on the record.

That matters twice. The investigator can escalate precisely rather than asking
for "more access", and the refusal log is what shows afterwards that the
response did not become the second incident — which is a question a regulator
will ask about an agent that read production during an outage.

## 4 · One admission set, six queries

Three are refused. Look at the third: same source and same fields as an allowed query, refused on volume alone.

### The skill — [`skills/secops/investigation-admission-rules/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/secops/investigation-admission-rules/SKILL.md)

```yaml
name: investigation-admission-rules
description: >-
  Decide and enforce which sources, fields and volumes an investigating agent
  may reach, before the investigation starts, and record every refusal. Use when
  granting an agent read access for incident response, when investigation
  queries touch personal data, or when an investigation's own access needs to be
  auditable.
allowed-tools: Read, Grep, Glob
```

# The investigation is the second-largest access grant in the incident

An investigating agent is handed broad read across the estate so it can find
the problem. Broad read across the estate is frequently what the problem *was*.

Admission rules decide, per investigation class and before anything runs, which
sources it may reach, which fields it may never see, and how much it may pull.

## When to use this

Before an agent-driven investigation touches production data, and whenever an
investigation class is defined. Also after the fact: the refusal log is the
evidence that the response did not become its own incident.

## Step-by-step

**1 — Classify the investigation first.** "Agent misuse" and "data exfiltration"
need different sources. One admission set for everything is no admission set.

**2 — Name the sources positively.** An allowlist. A denylist of sources is a
list of the ones somebody thought of.

**3 — Deny fields, not just sources.** The trace table is admissible; the
message body inside it usually is not.

**4 — Cap the volume.** The same query over the same fields is an investigation
at 400 rows and a copy at 90,000.

**5 — Record refusals as evidence, not errors.** A refusal carries the exact
query, so a human can grant it deliberately.

## Example

**Input** — one admission set and six queries, in
[`scripts/investigation_admission_rules.py`](scripts/investigation_admission_rules.py).

**Output** — the refusals from a real run:

```
bookings.db     owner_id,payment_card                8000  REFUSE — source not admitted for this class
agent.traces    agent,message_body                    900  REFUSE — denied field: message_body
gateway.logs    src,route                           90000  REFUSE — 90000 rows exceeds the 5000 cap
```

The third is refused on volume alone — same source, same fields as an allowed
query.

## Output contract

```json
{
  "allowed": [{"source": "str", "fields": ["str"], "rows": 0, "why": null}],
  "refused": [{"source": "str", "fields": ["str"], "rows": 0, "why": "str"}]
}
```

## Common edge cases

- **The agent needs the denied field to answer.** Then a human grants it, on
  the record — which is the outcome, not a failure of the rule.
- **Volume caps break a legitimate sweep.** Raise the cap for that class
  explicitly rather than removing it.
- **A source is admitted but joins to one that is not.** Enforce at the tool
  boundary, not on the query text.

## Failure modes

- **Granting once, broadly, "for the duration".** The duration is when the
  access is least supervised.
- **Silent drops.** A refusal nobody sees is indistinguishable from no data.
- **Rules written after the first investigation.** They will be written to
  permit whatever that one did.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/secops/investigation-admission-rules/scripts/investigation_admission_rules.py
SCRIPT = "skills/secops/investigation-admission-rules/scripts/investigation_admission_rules.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Three queries allowed and three refused — one on an inadmissible source, one on a denied field, and one on volume alone.

## Your turn

Write the admission set for a data-exfiltration investigation. It is not the same set, and working out why is the exercise.

---

**Next → [D3.3 · The context that makes agent triage work](https://spbreed.github.io/cyber-commons/lessons/D3.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D3.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D3.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*